# Validação da gold: os KPIs recalculados por outro caminho (etapas 4.2 a 4.4)

Confere se a gold e as Metric Views reproduzem o que se calcula **direto da silver**, sem usar nenhuma tabela gold para montar o número esperado. Cada verificação grava uma linha em `olist.quality.validacao_gold` (esperado × obtido × ok), e o notebook **falha no final** se alguma estiver reprovada.

**Por que outro caminho:** a gold não pode ser auditada pelo código que a produz. O esperado vem de uma consulta escrita de novo sobre `orders`, `order_items`, `order_status_events`, `order_reviews`, `products`, `category_translation`, `customers_historico` e `sellers_historico`; o obtido vem das tabelas gold e das Metric Views.

| # | Grupo | Verificações | Esperado |
|---|---|---|---|
| 1 | Linhas | `linhas_igual_silver` | cada tabela gold tem as linhas da sua origem (pedidos, itens, reviews, produtos, versões de cliente e de vendedor) |
| 2 | Granularidade | `granularidade_sem_duplicata` | uma linha por pedido, por item, por (review, pedido) e por chave substituta |
| 3 | Chaves | `fatos_sem_dimensao`, `versao_cobre_a_compra` | 0 fatos sem dimensão e 0 versões que não cobrem a data da compra |
| 4 | Totais | `gmv_bruto_itens`, `gmv_bruto_pedidos` | soma de preço + frete igual à da silver, em centavos |
| 5 | Derivações | `status_atual_igual_eventos`, `entrega_igual_evento`, `reviews_*` | 0 pedidos com status ou data de entrega diferente do recalculado; uma review por pedido |
| 6 | KPIs | `kpi_<nome>` | os 8 KPIs das Metric Views iguais aos recalculados da silver |
| 7 | Recortes | `recorte_<dimensão>` | 0 grupos (mês, categoria, vendedor, UF...) com valor diferente |
| 8 | Agregados (4.3) | `scorecard_*`, `daily_*`, `sla_*`, `atraso_review_*` | scorecard, KPIs diários, SLA e atraso × review iguais ao recalculado da silver; o critério 6 do desenho |
| 9 | Dashboard (4.4) | `dashboard_*` | o dashboard existe, todos os seus datasets executam e os KPIs dele são os mesmos das Metric Views |
| 10 | Idempotência | `idempotencia_<tabela>` | com a silver igual à da validação anterior, as tabelas gold não mudam |

**Escala dos valores:** a tabela guarda `esperado` e `obtido` como inteiros. Dinheiro vai em centavos (escala 100, tolerância 0), razões em milionésimos (escala 1.000.000) e dias em décimos de milésimo (escala 10.000); nas razões e nos dias a tolerância é de 1 unidade da escala, só para absorver o arredondamento.

## Configuração

Constantes, o identificador desta validação e a lista onde as verificações se acumulam.

In [0]:
# O QUE FAZ: define as constantes (catálogo, nome da tabela de resultados) e cria a lista onde as
#   verificações vão se acumulando. Nenhum dado é lido aqui.
#
# RECURSOS:
# - uuid.uuid4(): identificador aleatório único desta validação (agrupa as linhas dela).
# - f"...": f-string, insere variáveis no texto.
import uuid

from pyspark.sql import functions as F

CATALOG = "olist"
TABELA_RESULTADOS = f"{CATALOG}.quality.validacao_gold"
VALIDACAO_ID = str(uuid.uuid4())

resultados = []  # (entidade, verificacao, esperado, obtido, escala, ok, detalhe)
print(f"validacao_id: {VALIDACAO_ID}")

## Tabela de resultados e funções auxiliares

`registrar` guarda uma contagem (esperado e obtido inteiros). `registrar_valor` guarda um número com casas decimais já multiplicado pela escala. `n` roda uma consulta que devolve um número só. `gold` e `silver` montam nomes de tabela.

In [0]:
# O QUE FAZ: cria a tabela de resultados (se não existir) e declara as funções auxiliares.
#
# FUNÇÕES DECLARADAS:
# - gold(nome) / silver(nome): nome completo da tabela (olist.gold.x, olist.silver.x).
# - n(consulta): executa um SQL e devolve o primeiro valor da primeira linha como número Python.
# - registrar(entidade, verificacao, esperado, obtido, detalhe): contagem; ok = (esperado == obtido).
#   Sem esperado (None), ok fica nulo = "só registro" (as fotos da idempotência).
# - registrar_valor(entidade, verificacao, esperado, obtido, escala, tolerancia, detalhe): multiplica
#   os dois valores pela escala, arredonda para inteiro e aceita diferença de até "tolerancia" unidades.
#
# FUNÇÕES NATIVAS:
# - spark.sql(texto): executa SQL e devolve um DataFrame. .first()[0]: primeira coluna da primeira linha.
# - float(x): converte número ou Decimal em decimal de máquina. round(x): arredonda. int(x): inteiro.
# - abs(x): valor absoluto. float("nan") não é tratado: valor nulo vira 0 antes de registrar.
def gold(nome):
    return f"{CATALOG}.gold.{nome}"


def silver(nome):
    return f"{CATALOG}.silver.{nome}"


def n(consulta):
    valor = spark.sql(consulta).first()[0]
    return 0 if valor is None else valor


def registrar(entidade, verificacao, esperado, obtido, detalhe=""):
    ok = None if esperado is None else (esperado == obtido)
    resultados.append((entidade, verificacao, esperado, obtido, 1, ok, detalhe))
    marca = "  " if ok is None else ("OK" if ok else "XX")
    print(f"[{marca}] {entidade:20} {verificacao:34} esperado={esperado} obtido={obtido} {detalhe}")


def registrar_valor(entidade, verificacao, esperado, obtido, escala, tolerancia=1, detalhe=""):
    e = int(round(float(esperado or 0) * escala))
    o = int(round(float(obtido or 0) * escala))
    ok = abs(e - o) <= tolerancia
    resultados.append((entidade, verificacao, e, o, escala, ok, detalhe))
    marca = "OK" if ok else "XX"
    print(f"[{marca}] {entidade:20} {verificacao:34} esperado={esperado} obtido={obtido} {detalhe}")


spark.sql(f'''
CREATE TABLE IF NOT EXISTS {TABELA_RESULTADOS} (
  validacao_id  STRING    COMMENT 'Identificador da execução da validação',
  verificado_em TIMESTAMP COMMENT 'Momento da verificação',
  entidade      STRING    COMMENT 'Tabela ou métrica verificada',
  verificacao   STRING    COMMENT 'Nome da verificação',
  esperado      BIGINT    COMMENT 'Valor esperado (recalculado da silver), multiplicado pela escala',
  obtido        BIGINT    COMMENT 'Valor obtido na gold ou na Metric View, multiplicado pela escala',
  escala        BIGINT    COMMENT 'Multiplicador dos valores: 1 contagem, 100 centavos, 1000000 razão, 10000 dias',
  ok            BOOLEAN   COMMENT 'Verdadeiro se esperado = obtido (com a tolerância da verificação); nulo se não se aplica',
  detalhe       STRING    COMMENT 'Texto explicativo'
)
COMMENT 'Resultado da validação da gold: KPIs e tabelas recalculados direto da silver e comparados (etapa 4.2).'
''')

## Grupos 1 a 3 — linhas, granularidade e chaves

O esperado de cada contagem vem da **silver**. As verificações de chave usam `LEFT ANTI JOIN` (linhas do fato sem par na dimensão) e uma junção por intervalo escrita aqui de novo: a versão apontada por `customer_sk` / `seller_sk` tem que ser do mesmo cadastro e cobrir o instante da compra (`>= valido_de` e `< valido_ate`, ou `valido_ate` nulo).

In [0]:
# O QUE FAZ: registra as verificações dos grupos 1 a 3 (linhas, granularidade e chaves).
#
# FUNÇÕES NATIVAS (SQL):
# - count(*): linhas. count(DISTINCT a, b): combinações diferentes. count_if(cond): linhas que cumprem a condição.
# - LEFT ANTI JOIN: linhas da esquerda sem par na direita.
# - JOIN ... ON: junção por condição; aqui, a chave substituta e o intervalo de validade da versão.
# - IS NULL / IS NOT NULL: teste de nulo.
pares = (  # (tabela gold, origem na silver)
    ("fact_orders", "orders"),
    ("fact_order_items", "order_items"),
    ("fact_reviews", "order_reviews"),
    ("dim_product", "products"),
    ("dim_customer", "customers_historico"),
    ("dim_seller", "sellers_historico"),
)
for tabela_gold, origem in pares:
    registrar(tabela_gold, "linhas_igual_silver", n(f"SELECT count(*) FROM {silver(origem)}"), n(f"SELECT count(*) FROM {gold(tabela_gold)}"), f"origem silver.{origem}")

grao = (  # (tabela, expressão do grão)
    ("fact_orders", "order_id"),
    ("fact_order_items", "order_id, order_item_id"),
    ("fact_reviews", "review_id, order_id"),
    ("dim_product", "product_id"),
    ("dim_customer", "customer_sk"),
    ("dim_seller", "seller_sk"),
)
for tabela_gold, expr in grao:
    linhas = n(f"SELECT count(*) FROM {gold(tabela_gold)}")
    registrar(tabela_gold, "granularidade_sem_duplicata", linhas, n(f"SELECT count(DISTINCT {expr}) FROM {gold(tabela_gold)}"), f"grão ({expr})")

# fatos sem dimensão
registrar("fact_orders", "fatos_sem_dimensao", 0, n(f"SELECT count_if(customer_sk IS NULL) FROM {gold('fact_orders')}"), "customer_sk nulo")
registrar("fact_order_items", "fatos_sem_dimensao", 0, n(f"SELECT count_if(seller_sk IS NULL) FROM {gold('fact_order_items')}"), "seller_sk nulo")
registrar("fact_orders", "fatos_sem_dimensao", 0, n(f"SELECT count(*) FROM {gold('fact_orders')} f LEFT ANTI JOIN {gold('dim_customer')} d ON f.customer_sk = d.customer_sk"), "customer_sk sem dim_customer")
registrar("fact_order_items", "fatos_sem_dimensao", 0, n(f"SELECT count(*) FROM {gold('fact_order_items')} f LEFT ANTI JOIN {gold('dim_seller')} d ON f.seller_sk = d.seller_sk"), "seller_sk sem dim_seller")
registrar("fact_order_items", "fatos_sem_dimensao", 0, n(f"SELECT count(*) FROM {gold('fact_order_items')} f LEFT ANTI JOIN {gold('dim_product')} d ON f.product_id = d.product_id"), "product_id sem dim_product")
registrar("fact_orders", "fatos_sem_dimensao", 0, n(f"SELECT count(*) FROM {gold('fact_orders')} f LEFT ANTI JOIN {gold('dim_date')} d ON f.date_key = d.date_key"), "date_key sem dim_date")

# a versão apontada é do mesmo cadastro e cobre o instante da compra (junção por intervalo escrita de novo)
registrar("fact_orders", "versao_cobre_a_compra", 0, n(f'''
SELECT count(*) FROM {silver('orders')} o
JOIN {gold('fact_orders')} f ON f.order_id = o.order_id
JOIN {gold('dim_customer')} d ON d.customer_sk = f.customer_sk
WHERE NOT (d.customer_id = o.customer_id AND o.order_purchase_timestamp >= d.valido_de
           AND (o.order_purchase_timestamp < d.valido_ate OR d.valido_ate IS NULL))'''), "versão do cliente que não cobre a compra")
registrar("fact_order_items", "versao_cobre_a_compra", 0, n(f'''
SELECT count(*) FROM {silver('order_items')} i
JOIN {silver('orders')} o ON o.order_id = i.order_id
JOIN {gold('fact_order_items')} f ON f.order_id = i.order_id AND f.order_item_id = i.order_item_id
JOIN {gold('dim_seller')} d ON d.seller_sk = f.seller_sk
WHERE NOT (d.seller_id = i.seller_id AND o.order_purchase_timestamp >= d.valido_de
           AND (o.order_purchase_timestamp < d.valido_ate OR d.valido_ate IS NULL))'''), "versão do vendedor que não cobre a compra")

## Grupos 4 e 5 — totais e derivações

- **GMV bruto** (preço + frete de **todos** os itens, qualquer status): a silver, a `fact_order_items` e a `fact_orders` têm que somar o mesmo valor em centavos.
- **Status atual** recalculado com `max_by(status, event_sequence)`, que é outra forma de pegar "o evento de maior sequência" (a gold usa `row_number`). Conta pedidos com status diferente do da `fact_orders`.
- **Data de entrega**: o `event_ts` do evento `delivered`, comparada à coluna da `fact_orders`.
- **Reviews**: uma `eh_mais_recente` por pedido que tem review, e o número de reviews sem pedido igual ao da silver.

In [0]:
# O QUE FAZ: registra as verificações dos grupos 4 e 5 (totais e derivações), todas com o esperado
#   calculado só com tabelas da silver.
#
# FUNÇÕES NATIVAS (SQL):
# - sum(a + b): soma de preço + frete. max_by(valor, ordem): o valor da linha em que "ordem" é máxima.
# - min(CASE WHEN ... THEN coluna END): menor valor só entre as linhas que cumprem a condição.
# - <=>: igualdade que trata nulo = nulo como verdadeiro (usado para comparar datas de entrega nulas).
# - LEFT ANTI JOIN: linhas sem par. count(DISTINCT x): valores diferentes.
gmv_silver = n(f"SELECT sum(price + freight_value) FROM {silver('order_items')}")
registrar_valor("fact_order_items", "gmv_bruto_itens", gmv_silver, n(f"SELECT sum(item_total) FROM {gold('fact_order_items')}"), 100, 0, "soma de preço + frete, todos os itens")
registrar_valor("fact_orders", "gmv_bruto_pedidos", gmv_silver, n(f"SELECT sum(gmv_bruto) FROM {gold('fact_orders')}"), 100, 0, "soma de gmv_bruto, todos os pedidos")
registrar_valor("gold_seller_mensal", "gmv_so_pedidos_validos", n(f'''
SELECT sum(i.price + i.freight_value) FROM {silver('order_items')} i
JOIN (SELECT order_id, max_by(status, event_sequence) AS st FROM {silver('order_status_events')} GROUP BY order_id) e USING (order_id)
WHERE e.st NOT IN ('canceled', 'unavailable')''') + n(f'''
SELECT sum(i.price + i.freight_value) FROM {silver('order_items')} i
LEFT ANTI JOIN {silver('order_status_events')} e USING (order_id)'''), n(f"SELECT sum(gmv) FROM {gold('gold_seller_mensal')}"), 100, 0, "pedidos sem evento entram como válidos (ADR 0007)")

# status atual e entrega, recalculados dos eventos
ev = f'''
(SELECT order_id, max_by(status, event_sequence) AS st,
        min(CASE WHEN status = 'delivered' THEN event_ts END) AS ent
 FROM {silver('order_status_events')} GROUP BY order_id)'''
registrar("fact_orders", "status_atual_igual_eventos", 0, n(f'''
SELECT count(*) FROM {silver('orders')} o
LEFT JOIN {ev} e USING (order_id)
JOIN {gold('fact_orders')} f ON f.order_id = o.order_id
WHERE f.status_atual <> coalesce(e.st, 'desconhecido')'''), "pedidos com status diferente do recalculado")
registrar("fact_orders", "entrega_igual_evento", 0, n(f'''
SELECT count(*) FROM {silver('orders')} o
LEFT JOIN {ev} e USING (order_id)
JOIN {gold('fact_orders')} f ON f.order_id = o.order_id
WHERE NOT (f.order_delivered_customer_date <=> e.ent)'''), "pedidos com data de entrega diferente do evento delivered")

# reviews
registrar("fact_reviews", "reviews_uma_por_pedido", n(f"SELECT count(DISTINCT r.order_id) FROM {silver('order_reviews')} r JOIN {silver('orders')} o USING (order_id)"), n(f"SELECT count_if(eh_mais_recente) FROM {gold('fact_reviews')}"), "uma review que vale por pedido com review")
registrar("fact_reviews", "reviews_sem_pedido", n(f"SELECT count(*) FROM {silver('order_reviews')} r LEFT ANTI JOIN {silver('orders')} o USING (order_id)"), n(f"SELECT count_if(NOT pedido_existe) FROM {gold('fact_reviews')}"), "reviews sem pedido, fora das métricas")

## Grupo 6 — os 8 KPIs recalculados da silver

`KPIS_SILVER` é uma consulta nova, sobre a silver, que monta os pedidos com status, entrega, itens, review e cliente na data da compra e calcula todos os KPIs. Os mesmos números são pedidos às Metric Views com `MEASURE(...)`.

Um detalhe que diferencia os dois caminhos de propósito: a gold usa `row_number` para eleger o evento e a review mais recentes; aqui usa-se `max_by`. Os dias de entrega são calculados com `timestampdiff(SECOND, ...) / 86400` em vez de `unix_timestamp`.

In [0]:
# O QUE FAZ: monta a consulta-base sobre a silver (CTE "flags") e usa três finais diferentes: os KPIs
#   de pedido, a taxa de recompra e a concentração. Depois compara cada KPI com a Metric View.
#
# RECURSOS (SQL):
# - WITH nome AS (...): consulta nomeada; as CTEs ev, it, rv, base e flags se encadeiam.
# - max_by(x, struct(a, b, c)): valor de x na linha de maior (a, b, c), comparando campo a campo.
# - count_if, avg, percentile(coluna, 0.5): contagem condicional, média e mediana exata.
# - timestampdiff(SECOND, início, fim) / 86400.0: dias com fração entre dois instantes.
# - row_number() OVER (PARTITION BY mes ORDER BY gmv DESC, seller_id): posição do vendedor no mês.
# - str.format(c=...): troca {c} pelo catálogo (o SQL abaixo não usa outras chaves).
# - MEASURE(medida): lê a medida de uma Metric View.
BASE = '''
WITH ev AS (
  SELECT order_id, max_by(status, event_sequence) AS st,
         min(CASE WHEN status = 'delivered' THEN event_ts END) AS ent
  FROM {c}.silver.order_status_events GROUP BY order_id),
it AS (
  SELECT order_id, count(*) AS n, sum(price + freight_value) AS gmv
  FROM {c}.silver.order_items GROUP BY order_id),
rv AS (
  SELECT order_id, max_by(review_score, struct(review_answer_timestamp, review_creation_date, review_id)) AS sc
  FROM {c}.silver.order_reviews GROUP BY order_id),
base AS (
  SELECT o.order_id, o.order_purchase_timestamp AS t, o.order_estimated_delivery_date AS est,
         coalesce(ev.st, 'desconhecido') AS st, ev.ent, coalesce(it.n, 0) AS n,
         coalesce(it.gmv, 0) AS gmv, rv.sc, h.customer_unique_id AS uid, h.customer_state AS uf
  FROM {c}.silver.orders o
  LEFT JOIN ev USING (order_id) LEFT JOIN it USING (order_id) LEFT JOIN rv USING (order_id)
  LEFT JOIN {c}.silver.customers_historico h
    ON h.customer_id = o.customer_id AND o.order_purchase_timestamp >= h.__START_AT
   AND (o.order_purchase_timestamp < h.__END_AT OR h.__END_AT IS NULL)),
flags AS (
  SELECT *, (n > 0 AND st NOT IN ('canceled', 'unavailable')) AS valido,
         (st = 'delivered' AND ent IS NOT NULL) AS entregue
  FROM base)
'''.format(c=CATALOG)

kpis_pedido = spark.sql(BASE + '''
SELECT count(*) AS criados, count_if(valido) AS validos,
       sum(CASE WHEN valido THEN gmv END) AS gmv,
       count_if(st = 'canceled') AS cancelados,
       count_if(entregue) AS entregues,
       count_if(entregue AND to_date(ent) <= to_date(est)) AS no_prazo,
       avg(CASE WHEN entregue THEN timestampdiff(SECOND, t, ent) / 86400.0 END) AS lt_medio,
       percentile(CASE WHEN entregue THEN timestampdiff(SECOND, t, ent) / 86400.0 END, 0.5) AS lt_mediana,
       avg(CASE WHEN entregue THEN sc END) AS review
FROM flags''').first()

recompra = spark.sql(BASE + '''
SELECT count_if(c >= 2) AS recorrentes, count(*) AS clientes
FROM (SELECT uid, count(*) AS c FROM flags WHERE valido GROUP BY uid)''').first()

concentracao = spark.sql(BASE + '''
, vm AS (
  SELECT date_format(f.t, 'yyyy-MM') AS mes, i.seller_id, sum(i.price + i.freight_value) AS gmv
  FROM {c}.silver.order_items i JOIN flags f USING (order_id)
  WHERE f.valido GROUP BY 1, 2),
rk AS (SELECT *, row_number() OVER (PARTITION BY mes ORDER BY gmv DESC, seller_id) AS rn FROM vm)
SELECT sum(CASE WHEN rn <= 10 THEN gmv END) / sum(gmv) AS conc FROM rk'''.format(c=CATALOG)).first()

mv = spark.sql('''
SELECT MEASURE(pedidos_criados) AS criados, MEASURE(pedidos_validos) AS validos, MEASURE(gmv) AS gmv,
       MEASURE(ticket_medio) AS ticket, MEASURE(pedidos_cancelados) AS cancelados,
       MEASURE(taxa_cancelamento) AS tx_cancel, MEASURE(pedidos_entregues) AS entregues,
       MEASURE(pct_entregas_no_prazo) AS pct_prazo, MEASURE(lead_time_medio_dias) AS lt_medio,
       MEASURE(lead_time_mediana_dias) AS lt_mediana, MEASURE(review_score_medio) AS review,
       MEASURE(clientes_com_compra) AS clientes, MEASURE(clientes_recorrentes) AS recorrentes,
       MEASURE(taxa_recompra) AS recompra
FROM olist.gold.mv_pedidos''').first()
mv_itens = spark.sql("SELECT MEASURE(gmv) AS gmv, MEASURE(pedidos_validos) AS validos, MEASURE(ticket_medio) AS ticket FROM olist.gold.mv_itens").first()
mv_conc = spark.sql("SELECT MEASURE(concentracao_top10) AS conc FROM olist.gold.mv_concentracao").first()

k = kpis_pedido
registrar("mv_pedidos", "kpi_pedidos_criados", k.criados, mv.criados)
registrar("mv_pedidos", "kpi_pedidos_validos", k.validos, mv.validos)
registrar_valor("mv_pedidos", "kpi_gmv", k.gmv, mv.gmv, 100, 0, "KPI 1")
registrar_valor("mv_pedidos", "kpi_ticket_medio", float(k.gmv) / k.validos, mv.ticket, 100, 1, "KPI 2, escala em centavos, tolerância de 1 centavo")
registrar_valor("mv_pedidos", "kpi_taxa_cancelamento", k.cancelados / k.criados, mv.tx_cancel, 1000000, 1, "KPI 3")
registrar("mv_pedidos", "kpi_pedidos_entregues", k.entregues, mv.entregues)
registrar_valor("mv_pedidos", "kpi_pct_entregas_no_prazo", k.no_prazo / k.entregues, mv.pct_prazo, 1000000, 1, "KPI 4")
registrar_valor("mv_pedidos", "kpi_lead_time_medio", k.lt_medio, mv.lt_medio, 10000, 1, "KPI 5, dias")
registrar_valor("mv_pedidos", "kpi_lead_time_mediana", k.lt_mediana, mv.lt_mediana, 10000, 1, "KPI 5, dias")
registrar_valor("mv_pedidos", "kpi_review_score_medio", k.review, mv.review, 1000000, 1, "KPI 6")
registrar("mv_pedidos", "kpi_clientes_com_compra", recompra.clientes, mv.clientes)
registrar("mv_pedidos", "kpi_clientes_recorrentes", recompra.recorrentes, mv.recorrentes)
registrar_valor("mv_pedidos", "kpi_taxa_recompra", recompra.recorrentes / recompra.clientes, mv.recompra, 1000000, 1, "KPI 7")
registrar_valor("mv_concentracao", "kpi_concentracao_top10", concentracao.conc, mv_conc.conc, 1000000, 1, "KPI 8")
registrar_valor("mv_itens", "kpi_gmv_igual_mv_pedidos", mv.gmv, mv_itens.gmv, 100, 0, "o GMV tem que ser o mesmo nas duas Metric Views")
registrar("mv_itens", "kpi_pedidos_validos_igual_mv_pedidos", mv.validos, mv_itens.validos)

## Grupo 7 — os mesmos KPIs por recorte

Um total certo pode esconder um recorte errado. Aqui, para cada dimensão, uma consulta nova sobre a silver agrupa o valor e uma `FULL OUTER JOIN` com o resultado da Metric View conta os **grupos que divergem** (valor diferente, ou grupo que existe só de um lado). Esperado: 0.

| Recorte | Metric View | Medida |
|---|---|---|
| mês | `mv_pedidos` e `mv_itens` | `gmv` |
| UF do cliente | `mv_pedidos` | `gmv` |
| categoria | `mv_itens` | `gmv` |
| vendedor | `mv_itens` | `gmv` |
| UF do vendedor | `mv_itens` | `gmv` |
| mês | `mv_concentracao` | `concentracao_top10` |

In [0]:
# O QUE FAZ: para cada recorte, calcula o valor por grupo na silver e na Metric View e conta os grupos
#   que divergem.
#
# FUNÇÕES DECLARADAS:
# - divergencias(consulta_silver, consulta_mv, tolerancia): devolve (grupos comparados, grupos divergentes).
#   As duas consultas devolvem duas colunas: "grupo" e "valor".
#
# FUNÇÕES NATIVAS:
# - DataFrame.alias("s"): apelido para separar as colunas dos dois lados.
# - DataFrame.join(outro, "grupo", "full_outer"): mantém os grupos de qualquer lado.
# - F.coalesce(a, F.lit(0)): troca nulo por 0 (grupo sem valor = 0).
# - F.abs(coluna): valor absoluto. .where(cond).count(): conta as linhas que cumprem a condição.
# - F.col("s.valor"): coluna "valor" do lado com apelido "s".
def divergencias(consulta_silver, consulta_mv, tolerancia):
    s = spark.sql(consulta_silver).select("grupo", F.col("valor").cast("double").alias("valor")).alias("s")
    m = spark.sql(consulta_mv).select("grupo", F.col("valor").cast("double").alias("valor")).alias("m")
    j = s.join(m, "grupo", "full_outer")
    diferenca = F.abs(F.coalesce(F.col("s.valor"), F.lit(0.0)) - F.coalesce(F.col("m.valor"), F.lit(0.0)))
    return j.count(), j.where(diferenca > tolerancia).count()


# itens com o mês, o status e o cadastro do vendedor na data da compra, recalculados da silver
ITENS = BASE + '''
, itens AS (
  SELECT i.order_id, i.seller_id, i.product_id, f.t, f.valido, i.price + i.freight_value AS total,
         hs.seller_state AS uf_vendedor,
         coalesce(tr.product_category_name_english, p.product_category_name, 'desconhecida') AS categoria
  FROM {c}.silver.order_items i
  JOIN flags f USING (order_id)
  LEFT JOIN {c}.silver.sellers_historico hs
    ON hs.seller_id = i.seller_id AND f.t >= hs.__START_AT AND (f.t < hs.__END_AT OR hs.__END_AT IS NULL)
  LEFT JOIN {c}.silver.products p ON p.product_id = i.product_id
  LEFT JOIN {c}.silver.category_translation tr ON tr.product_category_name = p.product_category_name)
'''.format(c=CATALOG)

recortes = (
    ("mes_pedidos", BASE + "SELECT date_format(t, 'yyyy-MM') AS grupo, sum(CASE WHEN valido THEN gmv END) AS valor FROM flags GROUP BY 1",
     "SELECT ano_mes AS grupo, MEASURE(gmv) AS valor FROM olist.gold.mv_pedidos GROUP BY ano_mes", 0.004, "GMV por mês"),
    ("uf_cliente", BASE + "SELECT uf AS grupo, sum(CASE WHEN valido THEN gmv END) AS valor FROM flags GROUP BY 1",
     "SELECT uf_cliente AS grupo, MEASURE(gmv) AS valor FROM olist.gold.mv_pedidos GROUP BY uf_cliente", 0.004, "GMV por UF do cliente"),
    ("mes_itens", ITENS + "SELECT date_format(t, 'yyyy-MM') AS grupo, sum(CASE WHEN valido THEN total END) AS valor FROM itens GROUP BY 1",
     "SELECT ano_mes AS grupo, MEASURE(gmv) AS valor FROM olist.gold.mv_itens GROUP BY ano_mes", 0.004, "GMV por mês, no grão do item"),
    ("categoria", ITENS + "SELECT categoria AS grupo, sum(CASE WHEN valido THEN total END) AS valor FROM itens GROUP BY 1",
     "SELECT categoria AS grupo, MEASURE(gmv) AS valor FROM olist.gold.mv_itens GROUP BY categoria", 0.004, "GMV por categoria"),
    ("vendedor", ITENS + "SELECT seller_id AS grupo, sum(CASE WHEN valido THEN total END) AS valor FROM itens GROUP BY 1",
     "SELECT seller_id AS grupo, MEASURE(gmv) AS valor FROM olist.gold.mv_itens GROUP BY seller_id", 0.004, "GMV por vendedor"),
    ("uf_vendedor", ITENS + "SELECT uf_vendedor AS grupo, sum(CASE WHEN valido THEN total END) AS valor FROM itens GROUP BY 1",
     "SELECT uf_vendedor AS grupo, MEASURE(gmv) AS valor FROM olist.gold.mv_itens GROUP BY uf_vendedor", 0.004, "GMV por UF do vendedor"),
    ("concentracao_mes", ITENS + ''', vm AS (SELECT date_format(t, 'yyyy-MM') AS mes, seller_id, sum(total) AS gmv FROM itens WHERE valido GROUP BY 1, 2),
rk AS (SELECT *, row_number() OVER (PARTITION BY mes ORDER BY gmv DESC, seller_id) AS rn FROM vm)
SELECT mes AS grupo, sum(CASE WHEN rn <= 10 THEN gmv END) / sum(gmv) AS valor FROM rk GROUP BY mes''',
     "SELECT ano_mes AS grupo, MEASURE(concentracao_top10) AS valor FROM olist.gold.mv_concentracao GROUP BY ano_mes", 0.000002, "concentração por mês"),
)
for nome, consulta_silver, consulta_mv, tolerancia, texto in recortes:
    comparados, divergentes = divergencias(consulta_silver, consulta_mv, tolerancia)
    registrar("recorte", f"recorte_{nome}", 0, divergentes, f"{texto}; {comparados} grupos comparados")

## Grupo 8 — agregados da etapa 4.3

Cada agregado é recalculado da silver por uma consulta nova, que parte dos mesmos pedidos (`flags`) e dos pares (pedido, vendedor) dos itens.

- **Scorecard** (opção A, ADR 0008): a soma de pedidos, itens e GMV; os pedidos compartilhados; e, **por vendedor**, pedidos, cancelados, entregues, no prazo e com review (grupos que divergem, esperado 0).
- **Critério 6:** itens vendidos antes e depois da primeira mudança de endereço do vendedor, pelo instante da compra, nos vendedores que se mudaram.
- **KPIs diários:** totais e, por dia, pedidos criados e GMV.
- **SLA:** totais e, por mês e UF do cliente, entregues e entregues no prazo.
- **Atraso × review:** pedidos e nota média por faixa de atraso, com o corte das faixas escrito de novo (`timestampdiff(DAY, ...)`).

In [0]:
# O QUE FAZ: registra as verificações dos agregados da etapa 4.3 (scorecard, critério 6, KPIs diários,
#   SLA e atraso x review), com o esperado calculado só com tabelas da silver.
#
# RECURSOS (SQL):
# - SELECT DISTINCT order_id, seller_id: cada par (pedido, vendedor) uma vez (a "opção A" do scorecard).
# - row_number() / count(*) OVER (PARTITION BY seller_id ORDER BY valido_de): número da versão do
#   vendedor e quantas versões ele tem (na dim_seller, que lista todas, vendam ou não).
# - timestampdiff(DAY, início, fim): dias entre duas datas; CASE WHEN em cadeia monta as faixas.
# - concat(a, '|', b): junta textos; usado como "grupo" (mês|UF).
# - Uso de divergencias(...) (definida no grupo 7): conta grupos com valor diferente.
PARES = BASE + '''
, pares AS (SELECT DISTINCT order_id, seller_id FROM {c}.silver.order_items),
pf AS (SELECT p.seller_id, f.* FROM pares p JOIN flags f USING (order_id))
'''.format(c=CATALOG)
SC = gold("gold_seller_scorecard")

# scorecard: totais
registrar("gold_seller_scorecard", "scorecard_pares_pedido_vendedor", n(PARES + "SELECT count(*) FROM pares"), n(f"SELECT sum(pedidos) FROM {SC}"), "opção A: pedido em cada vendedor que tem item nele")
registrar("gold_seller_scorecard", "scorecard_itens", n(f"SELECT count(*) FROM {silver('order_items')}"), n(f"SELECT sum(itens) FROM {SC}"))
registrar_valor("gold_seller_scorecard", "scorecard_gmv", k.gmv, n(f"SELECT sum(gmv) FROM {SC}"), 100, 0, "o GMV não duplica nos pedidos compartilhados")
registrar("gold_seller_scorecard", "scorecard_pedidos_compartilhados", n(PARES + "SELECT count(*) FROM pares p JOIN (SELECT order_id FROM pares GROUP BY order_id HAVING count(*) > 1) m USING (order_id)"), n(f"SELECT sum(pedidos_compartilhados) FROM {SC}"), "pares cujo pedido tem mais de um vendedor")

# scorecard: por vendedor (soma das versões)
por_vendedor = (
    ("pedidos", "count(*)", "pedidos"),
    ("cancelados", "count_if(st = 'canceled')", "pedidos_cancelados"),
    ("entregues", "count_if(entregue)", "pedidos_entregues"),
    ("no_prazo", "count_if(entregue AND to_date(ent) <= to_date(est))", "entregues_no_prazo"),
    ("com_review", "count_if(entregue AND sc IS NOT NULL)", "pedidos_com_review"),
)
for nome, expr_silver, coluna in por_vendedor:
    comparados, divergentes = divergencias(
        PARES + f"SELECT seller_id AS grupo, {expr_silver} AS valor FROM pf GROUP BY seller_id",
        f"SELECT seller_id AS grupo, sum({coluna}) AS valor FROM {SC} GROUP BY seller_id", 0.5)
    registrar("gold_seller_scorecard", f"scorecard_por_vendedor_{nome}", 0, divergentes, f"{comparados} vendedores comparados")

# critério 6: itens antes e depois da primeira mudança de endereço (instante da compra)
c6 = spark.sql(BASE + '''
, h AS (SELECT seller_id, min(__END_AT) AS primeira_mudanca FROM {c}.silver.sellers_historico GROUP BY seller_id HAVING count(*) > 1)
SELECT count_if(f.t < h.primeira_mudanca) AS antes, count_if(f.t >= h.primeira_mudanca) AS depois,
       count(DISTINCT i.seller_id) AS vendedores
FROM {c}.silver.order_items i JOIN flags f USING (order_id) JOIN h USING (seller_id)'''.format(c=CATALOG)).first()
c6_gold = spark.sql(f'''
WITH d AS (SELECT seller_sk, row_number() OVER (PARTITION BY seller_id ORDER BY valido_de) AS num,
                  count(*) OVER (PARTITION BY seller_id) AS versoes FROM {gold('dim_seller')})
SELECT sum(CASE WHEN d.num = 1 THEN s.itens END) AS antes, sum(CASE WHEN d.num > 1 THEN s.itens END) AS depois,
       count(DISTINCT s.seller_id) AS vendedores
FROM {SC} s JOIN d USING (seller_sk) WHERE d.versoes > 1''').first()
registrar("gold_seller_scorecard", "criterio6_itens_antes_da_mudanca", c6.antes, c6_gold.antes, "vendas na cidade antiga, pelo instante da compra")
registrar("gold_seller_scorecard", "criterio6_itens_depois_da_mudanca", c6.depois, c6_gold.depois, "vendas na cidade nova")
registrar("gold_seller_scorecard", "criterio6_vendedores_que_mudaram_e_venderam", c6.vendedores, c6_gold.vendedores)

# KPIs diários
DK = gold("gold_daily_kpis")
registrar("gold_daily_kpis", "daily_pedidos_criados", k.criados, n(f"SELECT sum(pedidos_criados) FROM {DK}"))
registrar("gold_daily_kpis", "daily_pedidos_validos", k.validos, n(f"SELECT sum(pedidos_validos) FROM {DK}"))
registrar_valor("gold_daily_kpis", "daily_gmv", k.gmv, n(f"SELECT sum(gmv) FROM {DK}"), 100, 0)
registrar("gold_daily_kpis", "daily_pedidos_cancelados", k.cancelados, n(f"SELECT sum(pedidos_cancelados) FROM {DK}"))
registrar("gold_daily_kpis", "daily_pedidos_entregues", k.entregues, n(f"SELECT sum(pedidos_entregues) FROM {DK}"))
registrar("gold_daily_kpis", "daily_entregues_no_prazo", k.no_prazo, n(f"SELECT sum(entregues_no_prazo) FROM {DK}"))
registrar_valor("gold_daily_kpis", "daily_soma_lead_time", k.lt_medio * k.entregues, n(f"SELECT sum(soma_lead_time_dias) FROM {DK}"), 10000, 1, "soma de dias das entregas")
for nome, expr_silver, coluna in (
    ("pedidos_criados", "count(*)", "pedidos_criados"),
    ("gmv", "sum(CASE WHEN valido THEN gmv END)", "gmv"),
):
    comparados, divergentes = divergencias(
        BASE + f"SELECT CAST(to_date(t) AS STRING) AS grupo, {expr_silver} AS valor FROM flags GROUP BY 1",
        f"SELECT CAST(order_purchase_date AS STRING) AS grupo, {coluna} AS valor FROM {DK}", 0.004)
    registrar("gold_daily_kpis", f"daily_por_dia_{nome}", 0, divergentes, f"{comparados} dias comparados")

# SLA
SLA = gold("gold_delivery_sla")
registrar("gold_delivery_sla", "sla_pedidos_entregues", k.entregues, n(f"SELECT sum(pedidos_entregues) FROM {SLA}"))
registrar("gold_delivery_sla", "sla_entregues_no_prazo", k.no_prazo, n(f"SELECT sum(entregues_no_prazo) FROM {SLA}"))
registrar("gold_delivery_sla", "sla_entregues_atrasados", n(BASE + "SELECT count_if(entregue AND to_date(ent) > to_date(est)) FROM flags"), n(f"SELECT sum(entregues_atrasados) FROM {SLA}"))
for nome, expr_silver, coluna in (
    ("entregues", "count_if(entregue)", "pedidos_entregues"),
    ("no_prazo", "count_if(entregue AND to_date(ent) <= to_date(est))", "entregues_no_prazo"),
):
    comparados, divergentes = divergencias(
        BASE + f"SELECT concat(date_format(t, 'yyyy-MM'), '|', uf) AS grupo, {expr_silver} AS valor FROM flags GROUP BY 1",
        f"SELECT concat(ano_mes, '|', uf_cliente) AS grupo, {coluna} AS valor FROM {SLA}", 0.5)
    registrar("gold_delivery_sla", f"sla_por_mes_uf_{nome}", 0, divergentes, f"{comparados} grupos (mês e UF) comparados")

# atraso x review
FAIXAS = BASE + '''
, f2 AS (
  SELECT CASE WHEN d <= -1 THEN '1_adiantado' WHEN d = 0 THEN '2_no_dia' WHEN d <= 3 THEN '3_atraso_1_a_3_dias'
              WHEN d <= 7 THEN '4_atraso_4_a_7_dias' WHEN d <= 14 THEN '5_atraso_8_a_14_dias'
              ELSE '6_atraso_15_dias_ou_mais' END AS faixa, sc
  FROM (SELECT timestampdiff(DAY, to_date(est), to_date(ent)) AS d, sc FROM flags WHERE entregue AND sc IS NOT NULL))
'''.format(c=CATALOG)
AR = gold("gold_atraso_vs_review")
registrar("gold_atraso_vs_review", "atraso_review_total", n(FAIXAS + "SELECT count(*) FROM f2"), n(f"SELECT sum(pedidos) FROM {AR}"), "entregues com review")
for nome, expr_silver, coluna in (("pedidos", "count(*)", "pedidos"), ("nota_media", "avg(sc)", "review_score_medio")):
    comparados, divergentes = divergencias(
        FAIXAS + f"SELECT faixa AS grupo, {expr_silver} AS valor FROM f2 GROUP BY faixa",
        f"SELECT faixa_atraso AS grupo, {coluna} AS valor FROM {AR}", 0.000001)
    registrar("gold_atraso_vs_review", f"atraso_review_por_faixa_{nome}", 0, divergentes, f"{comparados} faixas comparadas")

## Grupo 9 — o dashboard mostra os mesmos números (etapa 4.4)

O critério de aceite diz que SQL, Metric View e dashboard dão o **mesmo número**. Esta verificação lê a definição do dashboard publicado (pelo SDK do Databricks), executa **cada consulta de dataset** dele e compara os KPIs com os das Metric Views, que por sua vez já foram comparados com o recálculo da silver. Como os contadores do dashboard só mostram a coluna do dataset (uma linha), o número na tela é o do dataset.

- `dashboard_existe`: o dashboard com o nome esperado está no workspace.
- `dashboard_datasets_executam`: todos os datasets devolvem ao menos uma linha.
- `dashboard_kpi_*`: o valor do dataset de KPIs é igual ao da Metric View.

In [0]:
# O QUE FAZ: localiza o dashboard pelo nome, lê os datasets dele, executa cada consulta e compara os KPIs
#   com os valores das Metric Views (variáveis mv e mv_conc, calculadas no grupo 6).
#
# FUNÇÕES DECLARADAS:
# - serialized_dashboard(nome): devolve o JSON (texto) da definição do dashboard com esse nome, ou None.
#
# FUNÇÕES NATIVAS:
# - WorkspaceClient(): cliente do SDK do Databricks; no notebook usa as credenciais do próprio usuário.
# - w.lakeview.list() / w.lakeview.get(id): lista os dashboards AI/BI e lê um deles (serialized_dashboard).
# - json.loads(texto): transforma o JSON em dicionário Python.
# - "".join(linhas): junta as linhas da consulta (queryLines) em um texto único.
# - spark.sql(texto).collect(): executa a consulta do dataset e traz as linhas (poucas).
import json

from databricks.sdk import WorkspaceClient

NOME_DASHBOARD = "Olist Lakehouse - Métricas Oficiais"


def serialized_dashboard(nome):
    w = WorkspaceClient()
    for d in w.lakeview.list():
        if d.display_name == nome:
            return w.lakeview.get(d.dashboard_id).serialized_dashboard
    return None


definicao = serialized_dashboard(NOME_DASHBOARD)
registrar("dashboard", "dashboard_existe", 1, 0 if definicao is None else 1, NOME_DASHBOARD)

if definicao is not None:
    datasets = {d["name"]: "".join(d["queryLines"]) for d in json.loads(definicao)["datasets"]}
    executam = 0
    linhas = {}
    for nome, consulta in datasets.items():
        resultado = spark.sql(consulta).collect()
        linhas[nome] = resultado
        executam += 1 if len(resultado) > 0 else 0
    registrar("dashboard", "dashboard_datasets_executam", len(datasets), executam, f"{len(datasets)} datasets")

    kpi = linhas["ds_kpis"][0]
    registrar_valor("dashboard", "dashboard_kpi_gmv", mv.gmv, kpi.gmv, 100, 0, "GMV do dashboard = Metric View")
    registrar_valor("dashboard", "dashboard_kpi_ticket_medio", mv.ticket, kpi.ticket_medio, 100, 1)
    registrar_valor("dashboard", "dashboard_kpi_taxa_cancelamento", mv.tx_cancel, kpi.taxa_cancelamento, 1000000, 1)
    registrar_valor("dashboard", "dashboard_kpi_pct_no_prazo", mv.pct_prazo, kpi.pct_no_prazo, 1000000, 1)
    registrar_valor("dashboard", "dashboard_kpi_lead_time_mediana", mv.lt_mediana, kpi.lead_time_mediana, 10000, 1)
    registrar_valor("dashboard", "dashboard_kpi_review_medio", mv.review, kpi.review_medio, 1000000, 1)
    registrar_valor("dashboard", "dashboard_kpi_taxa_recompra", mv.recompra, kpi.taxa_recompra, 1000000, 1)
    registrar_valor("dashboard", "dashboard_kpi_concentracao", mv_conc.conc, linhas["ds_concentracao"][0].concentracao_top10, 1000000, 1)

## Grupo 10 — idempotência da gold

Cada validação guarda uma "foto" das tabelas silver e gold (contagens e GMV). Se a **silver** não mudou desde a validação anterior, a **gold** também não pode ter mudado: o esperado de cada foto da gold é o valor da validação anterior. Se a silver mudou (chegou dado novo), a comparação é pulada e dita no texto, porque aí a gold deveria mudar mesmo.

Na primeira validação não existe "anterior", e as fotos só ficam registradas.

In [0]:
# O QUE FAZ: registra as fotos de contagem das tabelas e, se houver uma validação anterior com a mesma
#   silver, compara as fotos da gold com as dela.
#
# FUNÇÕES NATIVAS:
# - spark.sql(...).collect(): traz as linhas para o Python (poucas linhas aqui).
# - dict(...) e dict comprehension {chave: valor for ...}: monta um dicionário a partir das linhas.
# - ORDER BY verificado_em DESC LIMIT 1: a validação mais recente.
# - all(...): verdadeiro se todos os itens da lista são verdadeiros.
fotos_silver = {f"foto_silver_{t}": n(f"SELECT count(*) FROM {silver(t)}") for t in ("orders", "order_items", "order_status_events", "order_reviews", "customers_historico", "sellers_historico")}
fotos_gold = {f"foto_gold_{t}": n(f"SELECT count(*) FROM {gold(t)}") for t in ("dim_date", "dim_product", "dim_customer", "dim_seller", "fact_orders", "fact_order_items", "fact_reviews", "gold_seller_mensal", "gold_seller_scorecard", "gold_delivery_sla", "gold_daily_kpis", "gold_atraso_vs_review")}
fotos_gold["foto_gold_gmv_centavos"] = int(round(float(n(f"SELECT sum(item_total) FROM {gold('fact_order_items')}")) * 100))

anterior = spark.sql(f"SELECT validacao_id FROM {TABELA_RESULTADOS} WHERE verificacao LIKE 'foto_silver_%' ORDER BY verificado_em DESC LIMIT 1").collect()
fotos_antes = {}
if anterior:
    linhas = spark.sql(f"SELECT verificacao, obtido FROM {TABELA_RESULTADOS} WHERE validacao_id = '{anterior[0][0]}' AND verificacao LIKE 'foto_%'").collect()
    fotos_antes = {r[0]: r[1] for r in linhas}

for nome, valor in {**fotos_silver, **fotos_gold}.items():
    registrar("fotos", nome, None, valor)

if not fotos_antes:
    print("Primeira validação: sem foto anterior para comparar; as fotos ficaram registradas.")
elif not all(fotos_antes.get(nome) == valor for nome, valor in fotos_silver.items()):
    print("A silver mudou desde a validação anterior: a comparação de idempotência foi pulada.")
else:
    for nome, valor in fotos_gold.items():
        registrar("idempotencia", f"idempotencia_{nome.replace('foto_gold_', '')}", fotos_antes.get(nome), valor, "silver igual à da validação anterior")

## Gravação e resultado final

Grava todas as linhas em `quality.validacao_gold` (uma tabela de histórico: cada validação acrescenta as suas) e falha o notebook se alguma verificação estiver reprovada, para que um job de orquestração (Fase 7) pare no problema.

In [0]:
# O QUE FAZ: grava os resultados, mostra o resumo e interrompe com erro se houver verificação reprovada.
#
# FUNÇÕES NATIVAS:
# - spark.createDataFrame(linhas, schema): cria um DataFrame a partir de uma lista de tuplas.
# - DataFrame.withColumns({...}): acrescenta colunas.
# - DataFrame.select(...) / .write.mode("append").saveAsTable(nome): acrescenta as linhas à tabela.
#     * append nunca apaga o histórico; o validacao_id separa uma validação da outra.
# - display(df): mostra o DataFrame como tabela interativa.
# - raise RuntimeError(texto): interrompe a execução com erro.
df = (
    spark.createDataFrame(
        resultados,
        "entidade string, verificacao string, esperado long, obtido long, escala long, ok boolean, detalhe string",
    )
    .withColumns({"validacao_id": F.lit(VALIDACAO_ID), "verificado_em": F.current_timestamp()})
    .select("validacao_id", "verificado_em", "entidade", "verificacao", "esperado", "obtido", "escala", "ok", "detalhe")
)
df.write.mode("append").saveAsTable(TABELA_RESULTADOS)

reprovadas = [r for r in resultados if r[5] is False]
total = [r for r in resultados if r[5] is not None]
print(f"\n{len(total) - len(reprovadas)} de {len(total)} verificações OK")
display(df.where(F.col("ok") == False).select("entidade", "verificacao", "esperado", "obtido", "escala", "detalhe"))  # noqa: E712

if reprovadas:
    raise RuntimeError(f"{len(reprovadas)} verificação(ões) reprovada(s): veja a tabela {TABELA_RESULTADOS}.")